# Day S6 — 다중 서열 정렬과 프로파일: 여러 서열에서 보존 패턴 읽기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/supplement/day-S6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/supplement/day-S6.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 칸별 보존도와 consensus

In [ ]:
from collections import Counter

msa = {                              # 가상의 다중 정렬 (이미 정렬된 상태, '-'는 gap)
    "seqA": "MKTEGVLAAQ-EKTKQGV",
    "seqB": "MKTEGVMAAQ-EKTKQGV",
    "seqC": "MKSEGVVAAQ-EKSKQGV",
    "seqD": "MKTDGVLAASGEKTKEGV",
    "seqE": "MRSEGIVAAS-DKTREGV",
}
names = list(msa)
length = len(msa[names[0]])
assert all(len(seq) == length for seq in msa.values())

consensus = ""
marks = ""
for col in range(length):
    column = [msa[name][col] for name in names]
    letter, count = Counter(column).most_common(1)[0]
    consensus += letter
    if count == len(names) and letter != "-":
        marks += "*"                          # 모든 서열이 같은 글자
    elif count >= 4:
        marks += "+"                          # 5개 중 4개 이상이 같음
    else:
        marks += " "

for name in names:
    print(f"{name:9s} {msa[name]}")
print(f"{'':9s} {marks}")
print(f"{'consensus':9s} {consensus}")
print("완전히 보존된 칸:", marks.count("*"), "/", length)

### 예시 2 — PSSM을 만들어 서열 훑기

In [ ]:
import numpy as np
import pandas as pd

sites = [            # 어떤 단백질이 붙는 가상의 DNA 자리 8개 (이미 정렬됨)
    "TGACGT",
    "TGACGC",
    "TGATGT",
    "TGACGT",
    "CGACGT",
    "TGACAT",
    "TGACGT",
    "TTACGT",
]
BASES = "ACGT"
n_sites, width = len(sites), len(sites[0])

counts = pd.DataFrame(0, index=list(BASES), columns=range(1, width + 1))
for site in sites:
    for pos, base in enumerate(site, start=1):
        counts.loc[base, pos] += 1

PSEUDO = 1                                             # 한 번도 안 나온 글자도 확률 0이 되지 않게
freq = (counts + PSEUDO) / (n_sites + 4 * PSEUDO)
pssm = np.log2(freq / 0.25)                            # 배경 빈도 0.25와 비교한 log-odds

print(counts)
print()
print(pssm.round(2))

def pssm_score(window):
    return sum(pssm.loc[base, pos] for pos, base in enumerate(window, start=1))

target = "AAGCTGACGTTCATTGATGTCC"
print()
for start in range(len(target) - width + 1):
    window = target[start:start + width]
    s = pssm_score(window)
    if s > 3:
        print(f"위치 {start + 1:2d}  {window}  점수 {s:5.2f}")

### 예시 3 — progressive 정렬의 첫 단계: 누구부터 묶을까

In [ ]:
import numpy as np
import pandas as pd

def nw_align(a, b, match=1, mismatch=-1, gap=-2):
    """S4의 전역 정렬을 줄인 버전. 정렬된 두 문자열을 돌려준다."""
    n, m = len(a), len(b)
    F = np.zeros((n + 1, m + 1), dtype=int)
    F[:, 0] = gap * np.arange(n + 1)
    F[0, :] = gap * np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match if a[i - 1] == b[j - 1] else mismatch
            F[i, j] = max(F[i - 1, j - 1] + s, F[i - 1, j] + gap, F[i, j - 1] + gap)
    out_a, out_b, i, j = "", "", n, m
    while i > 0 or j > 0:
        s = match if i > 0 and j > 0 and a[i - 1] == b[j - 1] else mismatch
        if i > 0 and j > 0 and F[i, j] == F[i - 1, j - 1] + s:
            out_a, out_b, i, j = a[i - 1] + out_a, b[j - 1] + out_b, i - 1, j - 1
        elif i > 0 and F[i, j] == F[i - 1, j] + gap:
            out_a, out_b, i = a[i - 1] + out_a, "-" + out_b, i - 1
        else:
            out_a, out_b, j = "-" + out_a, b[j - 1] + out_b, j - 1
    return out_a, out_b

def distance(a, b):
    """정렬한 뒤 '다른 칸의 비율'을 거리로 쓴다."""
    x, y = nw_align(a, b)
    return sum(p != q for p, q in zip(x, y)) / len(x)

seqs = {                              # 정렬하기 전의 가상 서열 (gap 없음)
    "seqA": "MKTEGVLAAQEKTKQGV",
    "seqB": "MKTEGVMAAQEKTKQGV",
    "seqC": "MKSEGVVAAQEKSKQGV",
    "seqD": "MKTDGVLAASGEKTKEGV",
    "seqE": "MRSEGIVAASDKTREGV",
}
names = list(seqs)
D = pd.DataFrame(0.0, index=names, columns=names)
for x in names:
    for y in names:
        D.loc[x, y] = distance(seqs[x], seqs[y])
print(D.round(2))

pairs = [(D.loc[x, y], x, y) for i, x in enumerate(names) for y in names[i + 1:]]
print("\n가까운 쌍부터:")
for d, x, y in sorted(pairs)[:4]:
    print(f"  {x} – {y} 거리 {d:.2f}")